# Activation Steering: Causal Test of Probe Directions

**Goal**: Test whether the linear probe directions found in Phase 1 are *causal* — does adding the direction to the residual stream during generation actually change whether the model follows system vs user instructions?

**Method**: For each Condition C sample, generate with `alpha * direction` added at the best probe layer. Sweep alpha from negative (push toward user) to positive (push toward system) and measure SCR.

**Directions tested**: Probe weight vector (supervised) and CMD (class-mean difference, unsupervised).

**Modules**: `steer.py` (steering helpers), `data.py` (config, prompts), `probe.py` (probe results)

In [ ]:
import sys
from pathlib import Path

# Ensure phase1_linear_probing/ is on the import path regardless of kernel cwd
_PHASE1_DIR = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
if _PHASE1_DIR.name != "phase1_linear_probing":
    _PHASE1_DIR = _PHASE1_DIR / "phase1_linear_probing"
sys.path.insert(0, str(_PHASE1_DIR))

import importlib
import json
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer
import threadpoolctl
threadpoolctl.threadpool_limits(2, "blas")

import data as _data_mod, probe as _probe_mod, steer as _steer_mod
for _m in [_data_mod, _probe_mod, _steer_mod]:
    importlib.reload(_m)

from data import (
    ProbeConfig, find_repo_root, load_sync_env,
    load_results, prepare_condition_c,
    build_formatted_prompt,
    _load_nn_model, _cleanup_nn_model,
)
from probe import (
    load_results as load_probe_results, results_path, ProbeResult,
)
from steer import (
    load_steering_directions, steer_and_generate,
    score_steered_output, run_steering_sweep, compute_steered_scr,
    run_condition_comparison, save_experiment_manifest,
)

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
PROBE_CONFLICTS_4 = [
    "json_only_vs_plain",
    "list_bullets_vs_numbered",
    "past_vs_present_tense",
    "starting_word_hello_greetings",
]

cfg = ProbeConfig(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    label_mode="binary",
    token_positions=["last_prompt"],
    cv_mode="grouped",
    n_cv_folds=4,
    use_scaler=False,
    probe_C=1.0,
    batch_size=1,
    conflict_ids=PROBE_CONFLICTS_4,
    run_id="curated4-8b-v001",
)
load_sync_env(cfg.repo_root)
cfg.ensure_dirs()

# Steering parameters
BATCH_SIZE = 16
SEED = 42
ALPHAS = [3, 6, 10]
MAX_NEW_TOKENS = 512

print(f"Run ID       : {cfg.run_id}")
print(f"Run dir      : {cfg.run_dir}")
print(f"Device       : {cfg.device}")
print(f"Model        : {cfg.model_name}")
print(f"Alphas       : {ALPHAS}")
print(f"Batch size   : {BATCH_SIZE}")

In [ ]:
# ── Load probe results, find best layer ───────────────────────────────────────
pos = cfg.token_positions[0]
rpath = results_path(cfg.run_dir, cfg.cv_mode, cfg.use_scaler)
results = load_probe_results(rpath)
pr = results[pos]

best_layer = int(pr.cv_scores.loc[pr.cv_scores["roc_auc_mean"].idxmax(), "layer"])
best_auc = pr.cv_scores.loc[pr.cv_scores["roc_auc_mean"].idxmax(), "roc_auc_mean"]
print(f"Best layer   : {best_layer}")
print(f"Best AUC     : {best_auc:.3f}")

In [ ]:
# ── Load steering directions, print cosine similarity ─────────────────────────
directions = load_steering_directions(cfg.run_dir, pos, best_layer)

print(f"Directions loaded: {list(directions.keys())}")
print(f"Probe shape      : {directions['probe'].shape}")

if "cmd_overall" in directions:
    cos_sim = np.dot(directions["probe"], directions["cmd_overall"])
    print(f"Probe–CMD cosine : {cos_sim:.4f}")

if "cmd_per_constraint" in directions:
    cpc = directions["cmd_per_constraint"]
    sims = {k: np.dot(directions["probe"], v) for k, v in cpc.items()}
    print(f"\nProbe–constraint CMD cosine similarities:")
    for k, s in sorted(sims.items(), key=lambda x: -x[1]):
        print(f"  {k:<40} {s:+.4f}")

In [ ]:
# ── Load Condition C samples (all — no subsampling) ──────────────────────────
df_all = load_results(cfg.data_dir, cfg.model_name)
df_c = prepare_condition_c(df_all, cfg.label_mode, conflict_ids=cfg.conflict_ids)
df_c = df_c.sort_values("conflict_id").reset_index(drop=True)

print(f"Condition C samples: {len(df_c)} across {df_c['conflict_id'].nunique()} conflicts")
print(f"SCR: {(df_c['label'] == 'followed_system').mean():.3f}")

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
model_nn, n_layers = _load_nn_model(cfg)
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
print(f"Model loaded: {n_layers} layers")

In [ ]:
# ── Smoke test: alpha=0, single sample ────────────────────────────────────────
test_row = df_c.iloc[0]
test_prompt = build_formatted_prompt(
    tokenizer, test_row["system_prompt"], test_row["user_prompt"]
)

print(f"Conflict: {test_row['conflict_id']}")
print(f"Direction: {test_row['direction']}")
print(f"System: {test_row['system_prompt'][:80]}...")
print(f"User: {test_row['user_prompt'][:80]}...")
print(f"Prompt tokens: {len(tokenizer.encode(test_prompt))}")
print()

response = steer_and_generate(
    model_nn, tokenizer, test_prompt,
    directions["probe"], best_layer, alpha=0.0,
    max_new_tokens=MAX_NEW_TOKENS,
)

original = test_row["response"]
check_len = min(200, len(original))

print(f"Response (alpha=0):\n{response[:500]}")
print(f"\n--- Response length: {len(response)} chars ---")
print(f"\nOriginal Phase 0 response:\n{original[:500]}")
print(f"\n--- Original length: {len(original)} chars ---")
print(f"\nFirst {check_len} chars match: {response[:check_len] == original[:check_len]}")
assert response[:check_len] == original[:check_len], (
    f"alpha=0 response diverges within first {check_len} chars"
)

In [ ]:
# ── Run all conditions (phase0 / unsteered / tracer_only / steered) ───────────
steer_dir = cfg.run_dir / "steering"

# Build steering directions dict (exclude cmd_per_constraint — it's a nested dict)
steer_directions = {}
if "probe" in directions:
    steer_directions["probe"] = directions["probe"]
if "cmd_overall" in directions:
    steer_directions["cmd_overall"] = directions["cmd_overall"]

results = run_condition_comparison(
    model_nn, tokenizer, df_c,
    steering_directions=steer_directions,
    layer=best_layer,
    alphas=ALPHAS,
    max_new_tokens=MAX_NEW_TOKENS,
    batch_size=BATCH_SIZE,
    output_dir=steer_dir,
    seed=SEED,
)

save_experiment_manifest(
    steer_dir, seed=SEED, model_name=cfg.model_name,
    conflict_ids=list(cfg.conflict_ids),
    direction_names=list(steer_directions.keys()),
    alphas=ALPHAS, max_new_tokens=MAX_NEW_TOKENS,
    batch_size=BATCH_SIZE, layer=best_layer,
)

print(f"\nResults keys: {list(results.keys())}")
for k, v in results.items():
    print(f"  {k}: {len(v)} rows")

In [ ]:
# ── Divergence analysis: Phase0 vs Unsteered vs Tracer-only ───────────────────
# Baseline SCR comparison
for name in ["phase0", "unsteered", "tracer_only"]:
    scr = (results[name]["label"] == "followed_system").mean()
    print(f"{name:15s} SCR: {scr:.3f}")

# Deltas
phase0_scr = (results["phase0"]["label"] == "followed_system").mean()
unsteered_scr = (results["unsteered"]["label"] == "followed_system").mean()
tracer_scr = (results["tracer_only"]["label"] == "followed_system").mean()
print(f"\nPhase0 → Unsteered:   {unsteered_scr - phase0_scr:+.3f}  (backend difference)")
print(f"Unsteered → Tracer:   {tracer_scr - unsteered_scr:+.3f}  (nnsight overhead)")

# Per-sample label agreement
for a, b in [("phase0", "unsteered"), ("unsteered", "tracer_only")]:
    agree = (results[a]["label"].values == results[b]["label"].values).mean()
    print(f"{a} vs {b} label agreement: {agree:.1%}")

# Per-conflict breakdown
print(f"\n{'conflict_id':40s}  {'phase0':>7s} / {'unsteered':>9s} / {'tracer':>6s}")
print("-" * 70)
for cid in sorted(df_c["conflict_id"].unique()):
    row = []
    for name in ["phase0", "unsteered", "tracer_only"]:
        mask = results[name]["conflict_id"] == cid
        scr = (results[name].loc[mask, "label"] == "followed_system").mean()
        row.append(f"{scr:.3f}")
    print(f"  {cid:40s}  {' / '.join(row)}")

In [ ]:
# ── Steered SCR by alpha, per direction ───────────────────────────────────────
for dir_name in steer_directions:
    key = f"steered_{dir_name}"
    if key not in results:
        continue
    df_s = results[key]
    print(f"\n{dir_name} direction:")
    scr_table = df_s.groupby("alpha").apply(
        lambda g: (g["label"] == "followed_system").mean()
    )
    print(scr_table.to_string())

In [ ]:
# ── Cleanup model ─────────────────────────────────────────────────────────────
_cleanup_nn_model(model_nn, cfg.device)
del model_nn
print("Model cleaned up")

## Visualization

Load results from disk if needed (no GPU required for plotting).

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── SCR vs Alpha curves with baseline reference lines ─────────────────────────
fig = go.Figure()

colors = {"probe": "blue", "cmd_overall": "red"}

for dir_name in steer_directions:
    key = f"steered_{dir_name}"
    if key not in results:
        continue
    df_s = results[key]
    scr_by_alpha = df_s.groupby("alpha").apply(
        lambda g: (g["label"] == "followed_system").mean()
    )
    fig.add_trace(go.Scatter(
        x=scr_by_alpha.index.tolist(), y=scr_by_alpha.values.tolist(),
        mode="lines+markers", name=f"{dir_name} direction",
        line=dict(color=colors.get(dir_name, "green"), width=2),
        marker=dict(size=8),
    ))

# Baseline reference lines
baselines = [
    ("Phase 0", "phase0", "dot", "gray"),
    ("Unsteered", "unsteered", "dash", "orange"),
    ("Tracer-only", "tracer_only", "dashdot", "purple"),
]
for label, key, dash, color in baselines:
    scr = (results[key]["label"] == "followed_system").mean()
    fig.add_hline(
        y=scr, line_dash=dash, line_color=color,
        annotation_text=f"{label} SCR = {scr:.3f}",
    )

fig.update_layout(
    title=f"Activation Steering: SCR vs Alpha (Layer {best_layer})",
    xaxis_title="Alpha (steering strength)",
    yaxis_title="System Compliance Rate (SCR)",
    yaxis=dict(range=[0, 1]),
    template="plotly_white",
    width=800, height=500,
    legend=dict(x=0.02, y=0.98),
)
fig.show()

In [ ]:
# ── Per-conflict SCR heatmap ──────────────────────────────────────────────────
# Use probe direction for heatmap
sweep_probe = results.get("steered_probe", pd.DataFrame())
if len(sweep_probe) > 0:
    pivot = sweep_probe.copy()
    pivot["sys"] = (pivot["label"] == "followed_system").astype(int)
    heatmap_data = pivot.pivot_table(
        index="conflict_id", columns="alpha", values="sys", aggfunc="mean"
    )
    heatmap_data = heatmap_data.reindex(columns=sorted(heatmap_data.columns))

    fig = go.Figure(data=go.Heatmap(
        z=heatmap_data.values,
        x=[str(a) for a in heatmap_data.columns],
        y=heatmap_data.index.tolist(),
        colorscale="RdBu",
        zmid=0.5,
        zmin=0, zmax=1,
        text=np.round(heatmap_data.values, 2).astype(str),
        texttemplate="%{text}",
        colorbar=dict(title="SCR"),
    ))

    fig.update_layout(
        title=f"Per-Conflict SCR by Alpha (Probe Direction, Layer {best_layer})",
        xaxis_title="Alpha",
        yaxis_title="Conflict ID",
        template="plotly_white",
        height=max(400, len(heatmap_data) * 25 + 100),
        width=900,
    )
    fig.show()
else:
    print("No steered_probe results to plot")

In [ ]:
# ── Label distribution stacked bars ───────────────────────────────────────────
if len(sweep_probe) > 0:
    label_order = ["followed_system", "followed_user", "followed_both", "followed_neither"]
    bar_colors = {"followed_system": "#2166ac", "followed_user": "#b2182b",
                  "followed_both": "#92c5de", "followed_neither": "#f4a582"}

    fig = go.Figure()

    for label in label_order:
        counts = []
        for alpha in sorted(sweep_probe["alpha"].unique()):
            sub = sweep_probe[sweep_probe["alpha"] == alpha]
            counts.append((sub["label"] == label).sum() / len(sub))
        fig.add_trace(go.Bar(
            name=label,
            x=[str(a) for a in sorted(sweep_probe["alpha"].unique())],
            y=counts,
            marker_color=bar_colors.get(label, "gray"),
        ))

    fig.update_layout(
        barmode="stack",
        title=f"Label Distribution by Alpha (Probe Direction, Layer {best_layer})",
        xaxis_title="Alpha",
        yaxis_title="Fraction",
        yaxis=dict(range=[0, 1]),
        template="plotly_white",
        width=800, height=450,
        legend=dict(x=0.02, y=0.98),
    )
    fig.show()

In [ ]:
# ── Qualitative examples ──────────────────────────────────────────────────────
if len(sweep_probe) > 0:
    example_conflict = sweep_probe["conflict_id"].value_counts().index[0]

    print(f"=== Qualitative Examples: {example_conflict} ===\n")

    example_rows = sweep_probe[
        sweep_probe["conflict_id"] == example_conflict
    ].sort_values("alpha")

    # Show system/user prompts once
    first = example_rows.iloc[0]
    print(f"System: {first['system_prompt'][:120]}")
    print(f"User:   {first['user_prompt'][:120]}")
    print()

    # Show one example per alpha
    for alpha in sorted(example_rows["alpha"].unique()):
        row = example_rows[example_rows["alpha"] == alpha].iloc[0]
        print(f"--- alpha={row['alpha']:+.0f} | label={row['label']} ---")
        print(row["response"][:300])
        print()